# 04 - ConvLSTM U-Net temporal segmentation baseline

This notebook trains a five-frame temporal model while preserving the baseline preprocessing, center-frame masks, and official EchoNet TRAIN/VAL/TEST split. Each sample reads frames `t-2` through `t+2` directly from the original AVI and predicts only the mask for frame `t`.

In [ ]:
# Kaggle setup. Skip this cell when the environment already satisfies requirements.txt.
%pip install -q monai opencv-python-headless pandas matplotlib tqdm

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

# On Kaggle, set PROJECT_ROOT to the directory containing src/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import split_by_echonet_filelist
from src.temporal_dataset import EchoNetTemporalDataset, load_temporal_metadata
from src.temporal_model import build_convlstm_unet
from src.temporal_train import (
    evaluate_temporal,
    fit_temporal,
    get_temporal_loss,
    plot_temporal_history,
    save_json,
    save_temporal_predictions,
    write_experiment_log,
)
from src.utils import load_echonet_tables, set_seed

# Override these with Kaggle input paths when the datasets are mounted elsewhere.
RAW_DIR = Path(os.environ.get('ECHONET_RAW_DIR', PROJECT_ROOT / 'data' / 'raw' / 'EchoNet-Dynamic'))
PROCESSED_DIR = Path(os.environ.get('ECHONET_PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed'))
VIDEOS_DIR = RAW_DIR / 'Videos'
RUN_DIR = Path('/kaggle/working/outputs/runs/convlstm_unet') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
FIGURES_DIR = RUN_DIR / 'figures'
PREDICTIONS_DIR = FIGURES_DIR / 'predictions'

for directory in [RUN_DIR, CHECKPOINT_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Raw EchoNet directory: {RAW_DIR}')
print(f'Processed dataset directory: {PROCESSED_DIR}')
print(f'Persistent run directory: {RUN_DIR}')

## Configuration

Use `smoke` first to verify the full path from AVI loading through checkpoint and prediction creation. Use `full` for the directly comparable temporal experiment.

In [ ]:
RUN_MODE = 'smoke'  # change to 'full' for the complete experiment

SMOKE_CONFIG = {
    'run_mode': 'smoke',
    'seed': 42,
    'sequence_length': 5,
    'image_size': [112, 112],
    'epochs': 1,
    'batch_size': 4,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'max_train_samples': 32,
    'max_val_samples': 16,
    'max_test_samples': 16,
    'channels': [16, 32, 64, 128],
}

FULL_CONFIG = {
    'run_mode': 'full',
    'seed': 42,
    'sequence_length': 5,
    'image_size': [112, 112],
    'epochs': 50,
    # Five-frame backpropagation uses substantially more memory than the 2D baseline.
    'batch_size': 8,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'max_train_samples': None,
    'max_val_samples': None,
    'max_test_samples': None,
    'channels': [16, 32, 64, 128],
}

config = SMOKE_CONFIG if RUN_MODE == 'smoke' else FULL_CONFIG
save_json(config, RUN_DIR / 'config.json')
config

## Load processed masks and official EchoNet splits

No random fallback is used. The exact baseline helper `split_by_echonet_filelist()` assigns every labeled center frame to the official video-level partition.

In [ ]:
metadata_path = PROCESSED_DIR / 'metadata.csv'
assert metadata_path.exists(), 'Run notebook 02 or mount the reusable processed Kaggle Dataset.'
assert VIDEOS_DIR.exists(), f'Videos directory not found: {VIDEOS_DIR}'

samples = load_temporal_metadata(metadata_path)
file_list, _ = load_echonet_tables(RAW_DIR)
train_samples, val_samples, test_samples = split_by_echonet_filelist(samples, file_list)

matched_count = len(train_samples) + len(val_samples) + len(test_samples)
assert matched_count == len(samples), (
    f'{len(samples) - matched_count} samples did not match the official EchoNet split.'
)
assert min(len(train_samples), len(val_samples), len(test_samples)) > 0, 'Every official split must contain samples.'

if config['max_train_samples'] is not None:
    train_samples = train_samples[:config['max_train_samples']]
if config['max_val_samples'] is not None:
    val_samples = val_samples[:config['max_val_samples']]
if config['max_test_samples'] is not None:
    test_samples = test_samples[:config['max_test_samples']]

print(f'Total processed labeled frames: {len(samples):,}')
print(f'Train samples: {len(train_samples):,}')
print(f'Validation samples: {len(val_samples):,}')
print(f'Test samples: {len(test_samples):,}')

In [ ]:
dataset_kwargs = {
    'videos_dir': VIDEOS_DIR,
    'sequence_length': config['sequence_length'],
    'image_size': tuple(config['image_size']),
}
train_dataset = EchoNetTemporalDataset(train_samples, augment=True, **dataset_kwargs)
val_dataset = EchoNetTemporalDataset(val_samples, augment=False, **dataset_kwargs)
test_dataset = EchoNetTemporalDataset(test_samples, augment=False, **dataset_kwargs)

loader_kwargs = {
    'batch_size': config['batch_size'],
    'num_workers': config['num_workers'],
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': config['num_workers'] > 0,
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

# Shape verification: [B, T, C, H, W] input and [B, 1, H, W] target.
sample_batch = next(iter(train_loader))
print(f"Sequence batch shape: {tuple(sample_batch['sequence'].shape)}")
print(f"Mask batch shape: {tuple(sample_batch['mask'].shape)}")

## Train ConvLSTM U-Net

The optimizer and Dice+BCE objective match the baseline defaults. Checkpoints, history, and configuration are written incrementally to persistent Kaggle output storage.

In [ ]:
model = build_convlstm_unet(
    in_channels=1,
    out_channels=1,
    channels=tuple(config['channels']),
).to(device)
loss_fn = get_temporal_loss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay'],
)

history = fit_temporal(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=config['epochs'],
    output_dir=RUN_DIR,
)
plot_temporal_history(history, FIGURES_DIR / 'training_curves.png')
display(history.tail())

## Best-checkpoint validation and held-out test evaluation

In [ ]:
best_checkpoint_path = CHECKPOINT_DIR / 'best_model.pt'
best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])

val_metrics = evaluate_temporal(model, val_loader, loss_fn, device)
test_metrics = evaluate_temporal(model, test_loader, loss_fn, device)
save_json(test_metrics, RUN_DIR / 'test_metrics.json')

print('Best-checkpoint validation metrics:', val_metrics)
print('Held-out test metrics:', test_metrics)

prediction_count = save_temporal_predictions(
    model=model,
    loader=test_loader,
    device=device,
    output_dir=PREDICTIONS_DIR,
    max_examples=10,
)
write_experiment_log(
    RUN_DIR / 'experiment_log.md',
    config=config,
    val_metrics=val_metrics,
    test_metrics=test_metrics,
)
print(f'Saved prediction examples: {prediction_count}')

## Verify persistent Kaggle outputs

In [ ]:
required_outputs = [
    CHECKPOINT_DIR / 'best_model.pt',
    CHECKPOINT_DIR / 'final_model.pt',
    FIGURES_DIR / 'training_curves.png',
    RUN_DIR / 'history.csv',
    RUN_DIR / 'config.json',
    RUN_DIR / 'test_metrics.json',
    RUN_DIR / 'experiment_log.md',
]
missing_outputs = [path for path in required_outputs if not path.exists()]
assert not missing_outputs, f'Missing required outputs: {missing_outputs}'

processed_images = list((PROCESSED_DIR / 'images').glob('*.png'))
processed_masks = list((PROCESSED_DIR / 'masks').glob('*.png'))
prediction_files = list(PREDICTIONS_DIR.glob('*.png'))
assert len(processed_images) == len(processed_masks), 'Processed image-mask counts differ.'
assert len(prediction_files) >= min(10, len(test_dataset)), 'Expected qualitative predictions were not saved.'

all_run_files = [path for path in RUN_DIR.rglob('*') if path.is_file()]
print(f'Processed image count: {len(processed_images):,}')
print(f'Processed mask count: {len(processed_masks):,}')
print(f'Prediction figure count: {len(prediction_files):,}')
print(f'Total ConvLSTM output files: {len(all_run_files):,}')
print(f'Processed dataset available at: {PROCESSED_DIR.resolve()}')
print(f'Experiment outputs available at: {RUN_DIR.resolve()}')
for path in sorted(all_run_files):
    print(f'  - {path}')